# Corrupted Data Evaluation
Systematische Evaluation des Ensembles über verschiedene Corruption-Typen und Severity-Level

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Clone repository
!git clone https://github.com/deadPixelsGreta/xAI-proj-m-ws2526.git

In [3]:
# Setup project root
import sys, os
from pathlib import Path

def find_project_root(start: Path) -> Path:
    markers = {".git", "requirements.txt", "setup.py", "pyproject.toml"}
    root = None
    for parent in [start, *start.parents]:
        if any((parent / m).exists() for m in markers):
            root = parent
    return root or start

cloned_repo_name = "xAI-proj-m-ws2526"
cloned_repo_path = Path.cwd() / cloned_repo_name

if cloned_repo_path.is_dir():
    os.chdir(cloned_repo_path)

ROOT = find_project_root(Path.cwd()).resolve()
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print("cwd:", Path.cwd())
print("root on sys.path:", str(ROOT) in sys.path)

cwd: /content/xAI-proj-m-ws2526
root on sys.path: True


In [ ]:
# Install dependencies
!pip install -r requirements.txt --quiet

## Load Corrupted Dataset

In [5]:
# 1. Copy zip from Drive to local VM
!cp /content/drive/MyDrive/corrupted.zip /content/

# 2. Unzip directly to /content/
!unzip -q /content/corrupted.zip -d /content/

# 3. Move to datasets folder
!mv /content/corrupted/ /content/xAI-proj-m-ws2526/datasets/

# 4. Remove the zip to save space
!rm /content/corrupted.zip

## Systematische Evaluation über alle Corruption-Typen und Levels

In [ ]:
import subprocess
import re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import shutil

# Checkpoints
CHECKPOINT_DIR = "/content/drive/MyDrive/best_models_resnet"
CHECKPOINTS = [
    "best_resnet50-20260130-181509_epoch8.pth",
    "best_resnet18-20260107-144624_epoch15.pth",
    "copy-best_resnet34_color_seed42.pth"
]
# Corruption
corruption_types = ['gaussian_noise', 'pixelate']

# Severity levels
severity_levels = [1, 2, 3, 4, 5]

# Results storage
results = []

print("="*80)
print("Starting Systematic Corruption Evaluation")
print("="*80)

for corruption in corruption_types:
    print(f"\n{'='*80}")
    print(f"Corruption Type: {corruption.upper()}")
    print("="*80)

    for severity in severity_levels:
        print(f"\n{'-'*80}")
        print(f"Evaluating: {corruption} - Severity Level {severity}")
        print("-"*80)

        # Path to severity level
        data_dir = f"datasets/test/{corruption}/severity_{severity}"

        # Check if path exists
        if not Path(data_dir).exists():
            print(f"Skipping: {data_dir} does not exist")
            continue

        # Run evaluation
        cmd = [
            "python", "-m", "experiments.eval_corruptions.scripts.ensemble_inference",
            "--evaluate",
            "--data-dir", data_dir,
            "--split", "",
            "--checkpoints"
        ]
        # alle Checkpoint-Pfade
        cmd.extend([f"{CHECKPOINT_DIR}/{ckpt}" for ckpt in CHECKPOINTS])

        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
            output = result.stdout

            # Overall Accuracy
            accuracy_match = re.search(r'Overall Ensemble Accuracy:\s*(\d+\.\d+)%', output)
            if accuracy_match:
                accuracy = float(accuracy_match.group(1))
                results.append({
                    'Corruption': corruption,
                    'Severity': severity,
                    'Accuracy': accuracy
                })
                print(f"Overall Ensemble Accuracy: {accuracy:.2f}%")

            # Print results
            print("\n" + output)

        except subprocess.CalledProcessError as e:
            print(f"Error: {e}")
            print(e.stderr)

print("\n" + "="*80)
print("Evaluation Check")
print("="*80)

df = pd.DataFrame(results)

if not df.empty:
    # Pivot-Table
    pivot_df = df.pivot(index='Severity', columns='Corruption', values='Accuracy')

    print("\n" + "="*80)
    print(f"Final: Accuracy across Corruption Types and Severity Levels for Ensemble")
    print("="*80)
    print(pivot_df.to_string())

    # CSV
    df.to_csv('ensemble_3res_corruption_evaluation_results.csv', index=False)
    print(f"\nSave: ensemble_3res_corruption_evaluation_results.csv")

    # Plot
    plt.figure(figsize=(12, 6))
    for corruption in df['Corruption'].unique():
        subset = df[df['Corruption'] == corruption]
        plt.plot(subset['Severity'], subset['Accuracy'], marker='o', label=corruption)

    plt.xlabel('Severity Level')
    plt.ylabel('Accuracy (%)')
    plt.title('3 Models Resnets Aug Robustness across Corruption Types')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks([1, 2, 3, 4, 5])
    plt.savefig('ensemble_3res_corruption_evaluation_plot.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Generate: ensemble_3res_corruption_evaluation_plot.png")

    # Google Drive
    shutil.copy(f'ensemble_3res_corruption_evaluation_results.csv', '/content/drive/MyDrive/')
    shutil.copy(f'ensemble_3res_corruption_evaluation_plot.png', '/content/drive/MyDrive/')
    print("All saved to Google Drive")

else:
    print("No results to display.")
    print(f"Results list length: {len(results)}")

## Optional: Speichern der Ergebnisse auf Google Drive

In [ ]:
# Kopieren Sie die Ergebnisse zu Google Drive
!cp corruption_evaluation_results.csv /content/drive/MyDrive/
!cp corruption_evaluation_plot.png /content/drive/MyDrive/

print("Results to Google Drive")